In [14]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import skimage.io
from pathlib import Path
import math
import skimage.draw
import glob


ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

import importlib 
import utils.preprocessing as pp
import utils.visualization as viz
importlib.reload(pp)
importlib.reload(viz)

from utils.preprocessing import (
    load_coco_annotations,
    images_missing_annotations,
    split_summary,
)
from utils.visualization import (
    plot_annotation_distribution,
    plot_category_frequency,
    plot_sample_annotations,
    plot_mask_verification,
    plot_contrast_comparison,
    plot_tooth_sizes,
)


DATA_DIR  = ROOT_DIR / 'data'
IMG_DIR   = DATA_DIR / 'images'
ANN_DIR   = DATA_DIR / 'annotations'
VIZ_DIR   = ROOT_DIR / 'outputs' / 'visualizations'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120


## Dataset Overview

In [15]:
# Loading a random image annotation file
train_coco = load_coco_annotations(str(ANN_DIR/'train.json'))
val_coco = load_coco_annotations(str(ANN_DIR/'val.json'))
coco = train_coco

print(f"Keys: {list(coco.keys())}\n")

# Dataset overview
print("DATASET OVERVIEW")

print(f"Train images: {len(train_coco['images'])}")
print(f"Train annotations: {len(train_coco['annotations'])}")
print(f"Val images: {len(val_coco['images'])}")
print(f"Val annotations: {len(val_coco['annotations'])}")
print(f"Total categories: {len(train_coco['categories'])}")


# Sample image info
print("\nSAMPLE IMAGE INFO")
for k, v in coco['images'][0].items():
    print(f"{k}: {v}")

# Sample annotation info
print("\nSAMPLE ANNOTATION FORMAT")
for k, v in coco['annotations'][0].items():
    if k == 'segmentation':
        print(f"segmentation: {len(v)} polygon(s)")
        print(f"points: {len(v[0])//2} (x,y) pairs")
        print(f"format: [x1,y1,x2,y2,...] flat list")
        print(f"first 8 values: {v[0][:8]}")
    else:
        print(f"{k}: {v}")

Keys: ['images', 'annotations', 'categories']

DATASET OVERVIEW
Train images: 1597
Train annotations: 45426
Val images: 400
Val annotations: 11453
Total categories: 52

SAMPLE IMAGE INFO
id: 1
width: 2440
height: 1292
file_name: pan-00577.jpg

SAMPLE ANNOTATION FORMAT
id: 1
image_id: 1
category_id: 1
segmentation: 1 polygon(s)
points: 26 (x,y) pairs
format: [x1,y1,x2,y2,...] flat list
first 8 values: [1153, 737, 1140, 728, 1133, 721, 1130, 716]
area: 18583
bbox: [1128, 498, 109, 246]
iscrowd: False
width: 2440
height: 1292


In [17]:
print('CATEGORIES')
print(f"{'id':>4}  {'name':<12}  {'supercategory'}")
for c in coco['categories']:
    print(f"  {c['id']:>2}  {c['name']:<12}  {c['supercategory']}")

adult = [c for c in coco['categories'] if int(c['name'].split('-')[1]) <= 48]
deciduous = [c for c in coco['categories'] if int(c['name'].split('-')[1]) >= 51]
print(f"\nAdult permanent (11–48): {len(adult)} classes")
print(f"Deciduous/primary (51–85): {len(deciduous)} classes")

CATEGORIES
  id  name          supercategory
   1  tooth-11      tooth
   2  tooth-12      tooth
   3  tooth-13      tooth
   4  tooth-14      tooth
   5  tooth-15      tooth
   6  tooth-16      tooth
   7  tooth-17      tooth
   8  tooth-18      tooth
   9  tooth-21      tooth
  10  tooth-22      tooth
  11  tooth-23      tooth
  12  tooth-24      tooth
  13  tooth-25      tooth
  14  tooth-26      tooth
  15  tooth-27      tooth
  16  tooth-28      tooth
  17  tooth-31      tooth
  18  tooth-32      tooth
  19  tooth-33      tooth
  20  tooth-34      tooth
  21  tooth-35      tooth
  22  tooth-36      tooth
  23  tooth-37      tooth
  24  tooth-38      tooth
  25  tooth-41      tooth
  26  tooth-42      tooth
  27  tooth-43      tooth
  28  tooth-44      tooth
  29  tooth-45      tooth
  30  tooth-46      tooth
  31  tooth-47      tooth
  32  tooth-48      tooth
  33  tooth-51      tooth
  34  tooth-52      tooth
  35  tooth-53      tooth
  36  tooth-54      tooth
  37  tooth-55     

## MIssing annotations

In [19]:
for split_name, split in [('train', train_coco), ('val', val_coco)]:
    missing = images_missing_annotations(split)
    print(f"{split_name}: {len(split['images']) - len(missing)} with annotations, {len(missing)} without")
    for img in missing:
        print(f"  → {img['file_name']} (id={img['id']})")
    print()

train: 1591 with annotations, 6 without
  → pan-01929.jpg (id=545)
  → pan-06227.jpg (id=856)
  → pan-03173.jpg (id=1355)
  → pan-04632.jpg (id=1420)
  → pan-04787.jpg (id=1455)
  → pan-07155.jpg (id=1522)

val: 399 with annotations, 1 without
  → pan-01928.jpg (id=51)



In [18]:
summary = split_summary(train_coco, val_coco)
print(f"Train images:      {summary['train_images']}")
print(f"Train annotations: {summary['train_annotations']}")
print(f"Val   images:      {summary['val_images']}")
print(f"Val   annotations: {summary['val_annotations']}")
print(f"Overlap files:     {len(summary['overlap_files'])}  {'✅ clean' if not summary['leakage'] else '❌ LEAKAGE'}")
if summary['leakage']:
    for f in summary['overlap_files']:
        print(f"  → {f}")
print(f"Train categories:  {summary['train_categories']}/52")
print(f"Val   categories:  {summary['val_categories']}/52")
if summary['missing_from_val']:
    print(f"⚠️  Missing from val: {summary['missing_from_val']}")
else:
    print('✅ All 52 categories in both splits')

Train images:      1597
Train annotations: 45426
Val   images:      400
Val   annotations: 11453
Overlap files:     0  ✅ clean
Train categories:  52/52
Val   categories:  52/52
✅ All 52 categories in both splits
